# Exercise 03 — PCA and Model Selection

Guided notebook scaffold for the assignment. Fill each `TODO` and run the cells in order. The original, graded implementation belongs in `task1.py`, `task2.py`, and `task3.py`; keep their function signatures, imports, constants, and random seed unchanged. The notebook is an aid, not a replacement for submitting the required Python files and report.

Run from the exercise folder. The supplied plotting functions save figures in `plots/`. The housing data has 2,413 examples, nine features, and price targets in k$.

In [2]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

from data_utils import load_housing, train_test_split
from plotting import (
    plot_explained_variance,
    plot_pc_scatter,
    plot_polynomial_fits,
    plot_errors,
    plot_cv,
)

SEED = 2026
N_TRAIN = 100
N_VAL = 30
DEGREES = np.arange(1, 7)
N_FOLDS = 5

features, y = load_housing()
print(f"features: {features.shape}; prices: {y.shape}")
features.head()

features: (2413, 9); prices: (2413,)


,living_area,lot_area,basement_area,garage_area,year_built,overall_quality,bedrooms,bathrooms,fireplaces
0,153.8,2951.5,100.3,49.1,1960,6,3,1,2
1,83.2,1079.7,81.9,67.8,1961,5,2,1,0
2,123.5,1325.4,123.5,29.0,1958,6,3,1,0
3,196.0,1036.8,196.0,48.5,1968,7,3,2,2
4,151.3,1284.8,86.2,44.8,1997,5,3,2,1


## Task 1 — Principal Component Analysis

### 1.1 Pen-and-paper derivation

For the four centered 2D examples in the assignment: calculate the covariance matrix, derive its characteristic polynomial and eigenvalues, normalize the principal directions, then calculate the first-component explained-variance ratio. For K=1, calculate all scores and reconstructions, the mean squared reconstruction error, and compare it with the second eigenvalue.

In [3]:
def standardize_and_fit_pca(features, n_components=None):
    """Return standardized feature matrix X and a PCA fitted to X."""
    # TODO: fit StandardScaler to features and transform them to an (N, D) array.
    # TODO: fit PCA(n_components=n_components) to X.
    # Return X, pca.
    raise NotImplementedError

In [4]:
def load_scores():
    """Return PC1 scores with shape (N, 1), and prices with shape (N,)."""
    features, prices = load_housing()
    X, pca = standardize_and_fit_pca(features, n_components=1)
    # TODO: compute z = pca.transform(X), with shape (N, 1).
    # TODO: flip z if the living_area coefficient of PC1 is negative.
    return z, prices

In [5]:
print("Raw feature variances:")
print(features.var(ddof=0).round(1).to_string())

X, pca = standardize_and_fit_pca(features)
# TODO: get the explained variance ratio from pca.explained_variance_ratio_.
ratio = ...

variance_table = pd.DataFrame(
    {
        "eigenvalue": pca.explained_variance_,
        "ratio": ratio,
        "cumulative ratio": np.cumsum(ratio),
    },
    index=pd.RangeIndex(1, len(ratio) + 1, name="j"),
)
print(variance_table.round(3))

errors = []
for K in range(1, len(ratio) + 1):
    # TODO: fit PCA with K components to the standardized data.
    pca_K = ...
    # TODO: use the fitted PCA to reconstruct X.
    X_hat = ...
    # TODO: append mean squared reconstruction error across all N*D entries.
    errors.append(...)

print("Reconstruction MSE by K:", errors)
plot_explained_variance(ratio, errors)

print("PC1 loadings:")
print(pd.Series(pca.components_[0], index=features.columns).round(2))
plot_pc_scatter(pca.transform(X)[:, :2], y, ratio)

Raw feature variances:
living_area          2013.1
lot_area           583330.6
basement_area        1443.1
garage_area           350.9
year_built            869.2
overall_quality         1.8
bedrooms                0.7
bathrooms               0.3
fireplaces              0.4


NotImplementedError: 

**Interpretation to write up (Q1.1–Q1.4):** Could a large-scale raw feature dominate PCA, and why standardize? Compare reconstruction errors with eigenvalues and discuss the relationship from Task 1.1(b). Explain a practical choice of K for compression. Interpret the important PC1 loadings and what its score measures. Finally, explain how an unsupervised direction can nevertheless order houses by price.

## Task 2 — Overfitting and model evaluation

Fit polynomial regression on the single feature `z1`. Begin with the provided four-house toy example, then use the fixed 100-house training set and remaining houses as the test set.

In [ ]:
def polynomial_model(degree):
    """Create a polynomial-features + linear-regression pipeline."""
    # TODO: return a make_pipeline with PolynomialFeatures(degree) and LinearRegression.
    raise NotImplementedError


def train_and_evaluate(x_train, y_train, x_eval, y_eval, degree):
    """Fit degree-d polynomial regression and return evaluation-set MSE."""
    model = polynomial_model(degree)
    # TODO: fit the model on the training data.
    # TODO: predict on x_eval and return mean_squared_error(y_eval, predictions).
    raise NotImplementedError


X_TOY = np.array([[1], [2], [3], [4]])
Y_TOY = np.array([1, 2, 2, 3])
X_TOY_TEST = np.array([[5]])
Y_TOY_TEST = np.array([3.4])

# TODO: fit degrees 1 and 3, print coefficients, training MSE, and test squared error.
# TODO: print the degree-3 design matrix to discuss its shape and rank.

In [ ]:
z, y = load_scores()
rng = np.random.default_rng(SEED)
z_train, z_test, y_train, y_test = train_test_split(z, y, N_TRAIN, rng)
print(f"{len(y_train)} training houses, {len(y_test)} test houses")

train_mse, test_mse = [], []
for degree in DEGREES:
    # TODO: append training and test MSE for this degree using train_and_evaluate.
    ...

grid = np.linspace(z.min(), z.max(), 300).reshape(-1, 1)
# TODO: fit degrees 1, 2, and 6 on z_train and predict on grid; make a {degree: curve} dict.
curves = ...
plot_polynomial_fits(
    z_train.ravel(), y_train, z_test.ravel(), y_test, grid.ravel(), curves
)

# Holdout validation: use the same RNG stream after the initial train/test split.
z_tr, z_val, y_tr, y_val = train_test_split(z_train, y_train, N_TRAIN - N_VAL, rng)
val_mse = []
for degree in DEGREES:
    # TODO: append validation MSE for this degree.
    ...

# TODO: choose the degree with minimum validation MSE, then evaluate it on the test set.
d_holdout = ...
plot_errors(DEGREES, train_mse, test_mse, val_mse)

**Interpretation to write up (Q2.1–Q2.5):** Why can a cubic interpolate four distinct x-values? Compare training-based and test-based choices for the toy example and name the phenomenon. Report train/test MSE by degree and explain why training MSE cannot increase as model degree expands. Use the fit plot to discuss underfitting and overfitting. Explain why test MSE must not select the degree, what test metric should remain an unbiased final report, and how the small validation sample / a different split affects reliability.

## Task 3 — Model selection with cross-validation

Use the same 100-house train/test split as Task 2. Split the training examples into five shuffled folds with `random_state=0`. For each degree, train on four folds and evaluate on the held-out fold; average the five validation MSEs, select the minimum, refit on all 100 training houses, then evaluate on the test set.

In [ ]:
z, y = load_scores()
rng = np.random.default_rng(SEED)
z_train, z_test, y_train, y_test = train_test_split(z, y, N_TRAIN, rng)

kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
fold_mse = {}
for degree in DEGREES:
    mse_for_degree = []
    for idx_fold, (idx_train, idx_val) in enumerate(kfold.split(z_train)):
        # TODO: fit polynomial_model(degree) on z_train[idx_train], y_train[idx_train].
        # TODO: predict on z_train[idx_val] and append its MSE to mse_for_degree.
        ...
    fold_mse[degree] = mse_for_degree

table = pd.DataFrame.from_dict(
    fold_mse,
    orient="index",
    columns=[f"fold {k}" for k in range(1, N_FOLDS + 1)],
)
table["CV MSE"] = table.mean(axis=1)
print(table.rename_axis("degree").round(1))

# TODO: select the degree with minimum mean CV MSE.
d_cv = ...
# TODO: evaluate the selected degree on the untouched test set.
print(f"Selected degree: {d_cv}; test MSE: "
      f"{train_and_evaluate(z_train, y_train, z_test, y_test, d_cv):.1f}")
test_mse_cv = [
    train_and_evaluate(z_train, y_train, z_test, y_test, degree)
    for degree in DEGREES
]
plot_cv(DEGREES, table["CV MSE"].to_numpy(), test_mse_cv, d_cv)

**Interpretation to write up (Q3.1–Q3.4):** From the fold table, determine the holdout choice for each fold individually and compare it to the mean-CV choice. Compare CV to the Task 2 holdout choice and discuss which is more trustworthy. Explain why every degree must use the same folds, and when a single holdout split may be preferable to cross-validation.

## Submission / report checklist

- Include the Task 1.1 derivation and the results/tables requested in Tasks 1–3.
- Include the provided plots: PCA explained variance/reconstruction, PC score scatter, polynomial fits, Task 2 error curves, and cross-validation comparison.
- Answer Q1.1–Q1.4, Q2.1–Q2.5, and Q3.1–Q3.4 with interpretations of your own results.
- Start the report with a cover letter containing both team members' names and matriculation numbers.
- Include a short disclosure of the generative-AI tools used, how they helped, and their limitations.
- Submit all Python files and the report PDF; do not rename the Python files, zip them, or upload the `plots/` folder or dataset. The sheet lists Python 3.11+ and the versions in `requirements.txt` as the runtime requirements.

### Agent output:

Created a TODO-based notebook scaffold: Ex03-PCA-ModelSelection.ipynb. It covers PCA, polynomial regression, holdout validation, cross-validation, interpretation prompts, and the report checklist. The notebook’s JSON and all seven Python cells pass syntax checks. I couldn’t run its setup cell because the selected Python environment does not have scikit-learn installed; the project dependencies are listed in requirements.txt.

Note: the workspace contains task1.py, task2.py, and task3.py; there is no taskx.py. The notebook is a learning scaffold, not a replacement for filling in those files.

Steps to complete
Task 1 — PCA: Complete standardization, fit PCA, compute PC1 scores and explained-variance ratios, and calculate reconstruction MSE for each number of components. Do the small pen-and-paper PCA derivation and interpret the loadings, compression tradeoff, and relationship to price.
Task 2 — Overfitting and holdout: Implement polynomial regression and MSE evaluation. Work through the toy example, then compare degrees 1–6 using the fixed 100-house training set. Use a separate validation split to select the degree; reserve the test set for final evaluation.
Task 3 — Cross-validation: Evaluate each degree over five shuffled folds of the same training set, select using mean validation MSE, then report test performance.
Report and submit: Include the requested tables, plots, derivations, and answers to Q1.1–Q3.4. Add the cover letter and AI-use disclosure. The assignment says to submit the Python files and report PDF, not the plots folder or dataset. See the assignment instructions for details.
Time estimate: about 6–8 focused hours with theory knowledge and basic Python: roughly 1–2 hours for implementation and checks, 1–2 hours for derivations/results, and 3–4 hours for interpreting results and preparing the report. Allow extra time if you need to set up the Python environment or polish the report.